<a href="https://colab.research.google.com/github/khkk24/data_science_analist/blob/main/poubelle_Intelligente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("techsash/waste-classification-data")

print("Path to dataset files:", path)

100%|██████████| 427M/427M [00:03<00:00, 137MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/techsash/waste-classification-data/versions/1


In [4]:
!ls /root/.cache/kagglehub/datasets/techsash/waste-classification-data/versions/1

dataset  DATASET


In [5]:
import os

In [6]:
data_dir = "/root/.cache/kagglehub/datasets/techsash/waste-classification-data/versions/1/DATASET/"

In [7]:
train_dir = os.path.join(data_dir, "TRAIN")
test_dir = os.path.join(data_dir, "TEST")


In [8]:
os.listdir(train_dir)


['R', 'O']

In [9]:
train_dir_r = os.path.join(train_dir, "R")
train_dir_o = os.path.join(test_dir, "O")
test_dir_r = os.path.join(test_dir, "R")
test_dir_o = os.path.join(test_dir, "O")

In [10]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg


In [11]:
## ImageDataGenerator
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [12]:
train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

In [13]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),
    batch_size=20,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=20,
    class_mode='binary'
)

Found 22564 images belonging to 2 classes.
Found 2513 images belonging to 2 classes.


In [14]:

batch_images, batch_labels = next(iter(train_generator))

In [15]:
batch_images.shape

(20, 150, 150, 3)

In [16]:
batch_labels

array([1., 1., 0., 1., 0., 0., 1., 0., 1., 1., 0., 0., 0., 0., 1., 0., 1.,
       1., 1., 0.], dtype=float32)

In [17]:
train_generator.class_indices

{'O': 0, 'R': 1}

In [18]:
batch_images[4].shape

(150, 150, 3)

# MODELISER

In [19]:
import tensorflow as tf
from tensorflow.keras.models import Sequential

model = Sequential([
    # Extraction de caractéristiques
    tf.keras.layers.Conv2D(16, (3,3), activation='relu', input_shape=(150, 150, 3)),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(32, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    # applatir
    tf.keras.layers.Flatten(),

    # Dense
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [20]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 148, 148, 16)        │             448 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 74, 74, 16)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 72, 72, 32)          │           4,640 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 36, 36, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 41472)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 512)                 │      21,234,176 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │             513 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 21,239,777 (81.02 MB)

 Trainable params: 21,239,777 (81.02 MB)

 Non-trainable params: 0 (0.00 B)

In [21]:
model_ckp = tf.keras.callbacks.ModelCheckpoint(filepath='model.weights.h5',
                                               monitor='val_accuracy',
                                               mode='max',
                                               save_best_only=True,
                                               save_weights_only=True
                                               )
stop = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5)

In [22]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
    )

In [23]:
h = model.fit(x=train_generator,
              validation_data=test_generator,
              epochs=20,
              callbacks=[model_ckp, stop])

Epoch 1/20


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1129/1129 ━━━━━━━━━━━━━━━━━━━━ 37s 28ms/step - accuracy: 0.7914 - loss: 0.5959 - val_accuracy: 0.8683 - val_loss: 0.3337
Epoch 2/20
1129/1129 ━━━━━━━━━━━━━━━━━━━━ 30s 26ms/step - accuracy: 0.8562 - loss: 0.3427 - val_accuracy: 0.8945 - val_loss: 0.2722
Epoch 3/20
1129/1129 ━━━━━━━━━━━━━━━━━━━━ 28s 24ms/step - accuracy: 0.8827 - loss: 0.2779 - val_accuracy: 0.8806 - val_loss: 0.3227
Epoch 4/20
1129/1129 ━━━━━━━━━━━━━━━━━━━━ 28s 24ms/step - accuracy: 0.9323 - loss: 0.1783 - val_accuracy: 0.8671 - val_loss: 0.4039
Epoch 5/20
1129/1129 ━━━━━━━━━━━━━━━━━━━━ 28s 25ms/step - accuracy: 0.9666 - loss: 0.0932 - val_accuracy: 0.8723 - val_loss: 0.4876
Epoch 6/20
1129/1129 ━━━━━━━━━━━━━━━━━━━━ 29s 25ms/step - accuracy: 0.9861 - loss: 0.0448 - val_accuracy: 0.8782 - val_loss: 0.4949
Epoch 7/20
1129/1129 ━━━━━━━━━━━━━━━━━━━━ 40s 25ms/step - accuracy: 0.9931 - loss: 0.0284 - val_accuracy: 0.8858 - val_loss: 0.6131
